In [1]:
#!/usr/bin/env python3
"""
Prepare BMMC RNA-Protein data for RNA -> Protein prediction.

Compared with the previous version, this script is explicitly organized for
the task of predicting protein expression from RNA profiles.

Main changes
------------
1) Protein feature names are aligned to RNA gene symbols using the HGNC
   CD molecules table (group-471.csv), while preserving the original ADT names
   in .var metadata.
2) Split generation is redefined for RNA -> Protein prediction:
   - training supervision is built from paired cells only
   - RNA-only / protein-only cells are still tracked for semi-supervised use
   - validation query is RNA
   - validation target is protein
3) Exported file names now match the task direction clearly.

Outputs
-------
Base files:
- RNA_counts_qc_rnaprot.h5ad
- protein_counts_qc.h5ad
- feature_aligned_rna_protein.h5ad
- protein_gene_name_mapping.csv

Split files per ratio:
- results_ratio_loop_rna_to_protein/single_000/
- ...
Each ratio contains:
- train_rna_all.h5ad
- train_rna_paired.h5ad
- train_protein_paired.h5ad
- train_rna_only.h5ad
- train_protein_only.h5ad
- val_rna_query.h5ad
- val_true_protein.h5ad
- split_info.json

Recommended use
---------------
For a standard supervised RNA -> Protein model:
- Train input  : train_rna_paired.h5ad
- Train target : train_protein_paired.h5ad
- Val input    : val_rna_query.h5ad
- Val target   : val_true_protein.h5ad

If you want to exploit unpaired cells later:
- RNA-only cells are in train_rna_only.h5ad
- Protein-only cells are in train_protein_only.h5ad
"""

from __future__ import annotations

import json
import os
import random
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from sklearn.model_selection import train_test_split


# =========================
# 0. Parameters
# =========================
INPUT_H5AD = "/data5/zhangye/scMRDR/input/BMMC/raw_input/GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad"
OUTPUT_DIR = "/data5/zhangye/scMRDR/input/BMMC/preprocessed_input/RNA_Protein"
HGNC_CD_GENE_CSV = "/data5/zhangye/scMRDR/scripts/BMMC/CD_gene/group-471.csv"

SEED = 1234
VAL_FRAC = 0.2
SINGLE_FRACS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
RNA_KEEP_PROB = 0.5

BATCH_KEY = "batch"
MT_PREFIX = "MT-"

RNA_MIN_GENES = 200
RNA_MAX_GENES = 5000
RNA_MAX_COUNTS = 15000
RNA_MAX_MT = 20
RNA_MIN_CELLS_PER_GENE = 3

MIN_TRAIN_PAIRED_CELLS = 50
MIN_VAL_QUERY_CELLS = 20


# =========================
# 1. Utilities
# =========================
def set_seed(seed: int = 1234) -> None:
    random.seed(seed)
    np.random.seed(seed)


def ensure_dir(path: os.PathLike) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)


def safe_write_h5ad(adata: ad.AnnData, path: os.PathLike) -> None:
    adata = adata.copy()
    if isinstance(adata.X, np.matrix):
        adata.X = np.asarray(adata.X)
    for k in list(adata.layers.keys()):
        if isinstance(adata.layers[k], np.matrix):
            adata.layers[k] = np.asarray(adata.layers[k])
    adata.write(str(path))


def get_common_names(*arrays: Sequence[str]) -> List[str]:
    if len(arrays) == 0:
        return []
    common = set(arrays[0])
    for arr in arrays[1:]:
        common &= set(arr)
    return sorted(common)


def subset_and_copy(
    adata: ad.AnnData,
    cells: Sequence[str],
    features: Sequence[str] | None = None,
) -> ad.AnnData:
    out = adata[list(cells)].copy()
    if features is not None:
        out = out[:, list(features)].copy()
    return out


def require_obs_column(adata: ad.AnnData, key: str, fill_value: str = "batch0") -> None:
    if key not in adata.obs.columns:
        adata.obs[key] = fill_value


def assign_partial_modality(
    cells: Sequence[str],
    single_frac: float = 0.2,
    rna_keep_prob: float = 0.5,
):
    """
    Simulate partially unpaired data.

    For a fraction of cells (single_frac), one modality is hidden:
    - RNA with probability rna_keep_prob
    - Protein otherwise

    Returns
    -------
    dict with keys:
    - modality
    - paired_cells
    - rna_only_cells
    - prot_only_cells
    """
    cells = list(cells)
    n = len(cells)
    n_single = round(n * single_frac)

    single_cells = random.sample(cells, n_single) if n_single > 0 else []
    paired_cells = sorted(set(cells) - set(single_cells))

    modality = {c: "paired" for c in cells}
    if n_single > 0:
        for c in single_cells:
            modality[c] = "RNA" if random.random() < rna_keep_prob else "PROT"

    rna_only = sorted([c for c, m in modality.items() if m == "RNA"])
    prot_only = sorted([c for c, m in modality.items() if m == "PROT"])

    return {
        "modality": modality,
        "paired_cells": paired_cells,
        "rna_only_cells": rna_only,
        "prot_only_cells": prot_only,
    }


def clr_normalize(mat: np.ndarray) -> np.ndarray:
    """CLR normalization for ADT counts. Input shape: cells x features."""
    mat = np.asarray(mat, dtype=np.float32)
    mat = np.maximum(mat, 0.0)
    logp = np.log1p(mat)
    gm = np.exp(np.mean(logp, axis=1, keepdims=True))
    out = np.log1p(mat / np.clip(gm, 1e-8, None))
    return out.astype(np.float32)


def sanitize_feature_name(name: str) -> str:
    """Normalize protein marker strings before HGNC lookup."""
    x = str(name).strip()
    x = x.replace("_", "-")
    for suffix in ["-TotalSeqB", "-TotalSeqC", "-TotalSeqA", "-protein", "-Protein"]:
        if x.endswith(suffix):
            x = x[: -len(suffix)]
    return x.strip()


def build_cd_hgnc_lookup(df_hgnc: pd.DataFrame) -> Dict[str, str]:
    required_cols = ["Approved symbol", "Previous symbols", "Aliases"]
    for col in required_cols:
        if col not in df_hgnc.columns:
            raise ValueError(f"Missing required HGNC column: {col}")

    df_hgnc = df_hgnc.copy()
    df_hgnc[["Previous symbols", "Aliases"]] = df_hgnc[["Previous symbols", "Aliases"]].fillna("")

    lookup: Dict[str, str] = {}
    for _, row in df_hgnc.iterrows():
        approved = str(row["Approved symbol"]).strip()
        if not approved:
            continue

        candidates = {approved}
        candidates.update(s.strip() for s in str(row["Previous symbols"]).split(",") if s.strip())
        candidates.update(s.strip() for s in str(row["Aliases"]).split(",") if s.strip())

        for cand in candidates:
            lookup[sanitize_feature_name(cand)] = approved
    return lookup


def collapse_duplicate_features(
    adata: ad.AnnData,
    new_names: Sequence[str],
    keep: str = "mean",
) -> ad.AnnData:
    """
    Rebuild AnnData after renaming features.
    If multiple protein markers map to the same gene symbol, collapse them.
    """
    if keep not in {"mean", "sum"}:
        raise ValueError("keep must be 'mean' or 'sum'")

    new_names = [str(x) for x in new_names]
    X = adata.X.toarray() if sparse.issparse(adata.X) else np.asarray(adata.X)
    X_df = pd.DataFrame(X, index=adata.obs_names, columns=new_names)
    if keep == "mean":
        X_df = X_df.T.groupby(level=0).mean().T
    else:
        X_df = X_df.T.groupby(level=0).sum().T

    out = ad.AnnData(
        X=X_df.values.astype(np.float32),
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=X_df.columns),
    )
    out.uns = dict(adata.uns)
    out.obsm = {k: v.copy() if hasattr(v, "copy") else v for k, v in adata.obsm.items()}
    out.obsp = {k: v.copy() if hasattr(v, "copy") else v for k, v in adata.obsp.items()}

    for layer_name, layer in adata.layers.items():
        layer_arr = layer.toarray() if sparse.issparse(layer) else np.asarray(layer)
        layer_df = pd.DataFrame(layer_arr, index=adata.obs_names, columns=new_names)
        if keep == "mean":
            layer_df = layer_df.T.groupby(level=0).mean().T
        else:
            layer_df = layer_df.T.groupby(level=0).sum().T
        out.layers[layer_name] = layer_df.values.astype(np.float32)

    return out


def align_protein_names_to_rna(
    prot: ad.AnnData,
    rna_var_names: Sequence[str],
    hgnc_csv: os.PathLike,
) -> Tuple[ad.AnnData, pd.DataFrame]:
    """
    Map protein/CD feature names to RNA gene symbols using the HGNC CD molecules table.
    Original ADT names are preserved in .var metadata.
    """
    prot = prot.copy()
    rna_name_set = {str(x) for x in rna_var_names}

    if not Path(hgnc_csv).exists():
        print(f"[Warning] HGNC file not found: {hgnc_csv}")
        print("          Protein names will be kept as-is.")
        mapping_df = pd.DataFrame({
            "original_protein_name": prot.var_names.astype(str),
            "sanitized_name": [sanitize_feature_name(x) for x in prot.var_names.astype(str)],
            "mapped_gene_symbol": prot.var_names.astype(str),
            "matched_to_rna": [x in rna_name_set for x in prot.var_names.astype(str)],
        })
        prot.var["protein_name_raw"] = prot.var_names.astype(str)
        prot.var["protein_name_sanitized"] = [sanitize_feature_name(x) for x in prot.var_names.astype(str)]
        prot.var["mapped_gene_symbol"] = prot.var_names.astype(str)
        prot.var["matched_to_rna"] = [x in rna_name_set for x in prot.var_names.astype(str)]
        return prot, mapping_df

    df_hgnc = pd.read_csv(hgnc_csv)
    lookup = build_cd_hgnc_lookup(df_hgnc)

    original_names = prot.var_names.astype(str).tolist()
    sanitized_names = [sanitize_feature_name(x) for x in original_names]
    mapped_names = [lookup.get(x, x) for x in sanitized_names]
    matched_to_rna = [x in rna_name_set for x in mapped_names]

    mapping_df = pd.DataFrame({
        "original_protein_name": original_names,
        "sanitized_name": sanitized_names,
        "mapped_gene_symbol": mapped_names,
        "matched_to_rna": matched_to_rna,
    })

    prot.var["protein_name_raw"] = original_names
    prot.var["protein_name_sanitized"] = sanitized_names
    prot.var["mapped_gene_symbol"] = mapped_names
    prot.var["matched_to_rna"] = matched_to_rna

    # For feature names used during training/export, use mapped names and collapse duplicates.
    prot = collapse_duplicate_features(prot, mapped_names, keep="mean")

    agg_map = mapping_df.groupby("mapped_gene_symbol", as_index=False).agg({
        "original_protein_name": lambda x: " | ".join(sorted(set(map(str, x)))),
        "sanitized_name": lambda x: " | ".join(sorted(set(map(str, x)))),
        "matched_to_rna": "max",
    })
    agg_map = agg_map.set_index("mapped_gene_symbol")

    prot.var["protein_name_raw"] = prot.var_names.map(agg_map["original_protein_name"].to_dict())
    prot.var["protein_name_sanitized"] = prot.var_names.map(agg_map["sanitized_name"].to_dict())
    prot.var["mapped_gene_symbol"] = prot.var_names.astype(str)
    prot.var["matched_to_rna"] = (
        prot.var_names.map(agg_map["matched_to_rna"].to_dict()).fillna(False).astype(bool)
    )

    return prot, mapping_df


def save_split_h5ads(
    outdir: os.PathLike,
    train_rna_all: ad.AnnData,
    train_rna_paired: ad.AnnData,
    train_protein_paired: ad.AnnData,
    train_rna_only: ad.AnnData,
    train_protein_only: ad.AnnData,
    val_rna_query: ad.AnnData,
    val_true_protein: ad.AnnData,
) -> None:
    outdir = Path(outdir)
    safe_write_h5ad(train_rna_all, outdir / "train_rna_all.h5ad")
    safe_write_h5ad(train_rna_paired, outdir / "train_rna_paired.h5ad")
    safe_write_h5ad(train_protein_paired, outdir / "train_protein_paired.h5ad")
    safe_write_h5ad(train_rna_only, outdir / "train_rna_only.h5ad")
    safe_write_h5ad(train_protein_only, outdir / "train_protein_only.h5ad")
    safe_write_h5ad(val_rna_query, outdir / "val_rna_query.h5ad")
    safe_write_h5ad(val_true_protein, outdir / "val_true_protein.h5ad")


# =========================
# 2. Read and preprocess
# =========================
def read_bmmc_cite(input_h5ad: os.PathLike) -> Tuple[ad.AnnData, ad.AnnData]:
    adata = sc.read_h5ad(str(input_h5ad))
    if "counts" in adata.layers:
        adata.X = adata.layers["counts"].copy()

    if "feature_types" in adata.var.columns:
        rna = adata[:, adata.var["feature_types"].astype(str) == "GEX"].copy()
        prot = adata[:, adata.var["feature_types"].astype(str) == "ADT"].copy()
    else:
        raise ValueError("feature_types not found; please split manually based on your file format.")

    rna.obsm = {k: rna.obsm[k] for k in ["GEX_X_pca", "GEX_X_umap"] if k in rna.obsm}
    prot.obsm = {k: prot.obsm[k] for k in ["ADT_X_pca", "ADT_X_umap", "ADT_isotype_controls"] if k in prot.obsm}
    return rna, prot


def preprocess_rna(rna: ad.AnnData) -> ad.AnnData:
    rna = rna.copy()
    require_obs_column(rna, BATCH_KEY, "batch0")

    rna.var["mt"] = rna.var_names.str.startswith(MT_PREFIX)
    sc.pp.calculate_qc_metrics(rna, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

    sc.pp.filter_genes(rna, min_cells=RNA_MIN_CELLS_PER_GENE)
    sc.pp.filter_cells(rna, min_genes=RNA_MIN_GENES)
    rna = rna[rna.obs["n_genes_by_counts"] <= RNA_MAX_GENES, :].copy()
    rna = rna[rna.obs["total_counts"] <= RNA_MAX_COUNTS, :].copy()
    rna = rna[rna.obs["pct_counts_mt"] < RNA_MAX_MT, :].copy()

    rna.layers["counts"] = rna.X.copy()
    sc.pp.normalize_total(rna)
    sc.pp.log1p(rna)
    sc.pp.highly_variable_genes(rna, batch_key=BATCH_KEY, min_mean=0.02, max_mean=4, min_disp=0.5)
    return rna


def preprocess_protein(prot: ad.AnnData) -> ad.AnnData:
    prot = prot.copy()
    require_obs_column(prot, BATCH_KEY, "batch0")

    prot.layers["counts"] = prot.X.copy()
    prot.X = clr_normalize(prot.X.toarray() if sparse.issparse(prot.X) else prot.X)

    # Keep all protein markers; the panel is already small.
    prot.var["highly_variable"] = True
    return prot


def build_feature_aligned(rna_qc: ad.AnnData, prot_qc: ad.AnnData) -> ad.AnnData:
    """
    Bookkeeping file only.
    For RNA -> Protein training, do not use this as a 'common feature space' input.
    """
    union_features = sorted(set(rna_qc.var_names.astype(str)) | set(prot_qc.var_names.astype(str)))
    adata = ad.concat([rna_qc, prot_qc], join="outer", label="modality", keys=["rna", "protein"])
    adata.uns["rna_hvg"] = list(rna_qc.var_names[rna_qc.var["highly_variable"]].astype(str))
    adata.uns["prot_hvg"] = list(prot_qc.var_names.astype(str))
    adata.uns["rna_nz"] = sorted(set(union_features) & set(rna_qc.var_names.astype(str)))
    adata.uns["prot_nz"] = sorted(set(union_features) & set(prot_qc.var_names.astype(str)))
    adata = adata[:, union_features].copy()
    return adata


# =========================
# 3. Split generation
# =========================
def generate_splits(
    rna_qc_path: os.PathLike,
    prot_qc_path: os.PathLike,
    out_root: os.PathLike,
    seed: int = 1234,
    val_frac: float = 0.2,
    single_fracs: Sequence[float] = (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    rna_keep_prob: float = 0.5,
):
    """
    Generate splits explicitly for RNA -> Protein prediction.

    Training
    --------
    - paired cells: supervised RNA -> protein learning
    - RNA-only cells: optional extra unlabeled RNA
    - protein-only cells: optional extra unlabeled protein

    Validation
    ----------
    - query is RNA
    - target is protein
    """
    set_seed(seed)

    rna = sc.read_h5ad(str(rna_qc_path))
    prot = sc.read_h5ad(str(prot_qc_path))

    common_cells = get_common_names(rna.obs_names.tolist(), prot.obs_names.tolist())
    if len(common_cells) == 0:
        raise ValueError("No common cells between RNA and protein after preprocessing.")

    rna = rna[common_cells].copy()
    prot = prot[common_cells].copy()

    train_cells, val_cells = train_test_split(
        common_cells,
        test_size=val_frac,
        random_state=seed,
        shuffle=True,
    )
    train_cells = sorted(train_cells)
    val_cells = sorted(val_cells)

    summaries = []
    out_root = Path(out_root)
    ensure_dir(out_root)

    for frac in single_fracs:
        ratio_label = f"single_{int(round(frac * 100)):03d}"
        outdir = out_root / ratio_label
        ensure_dir(outdir)

        set_seed(seed + int(round(frac * 1000)))

        train_assign = assign_partial_modality(train_cells, single_frac=frac, rna_keep_prob=rna_keep_prob)
        val_assign = assign_partial_modality(val_cells, single_frac=frac, rna_keep_prob=rna_keep_prob)

        train_paired = train_assign["paired_cells"]
        train_rna_only = train_assign["rna_only_cells"]
        train_prot_only = train_assign["prot_only_cells"]

        val_paired = val_assign["paired_cells"]
        val_rna_only = val_assign["rna_only_cells"]
        val_prot_only = val_assign["prot_only_cells"]

        # Supervised training pairs
        train_rna_paired_cells = train_paired
        train_protein_paired_cells = train_paired

        # Optional extra modality-specific training sets
        train_rna_all_cells = sorted(train_paired + train_rna_only)

        # RNA -> Protein validation:
        # query is RNA-visible cells; true target is their held-out protein values.
        # Protein-only validation cells cannot be queried by RNA, so exclude them.
        val_rna_query_cells = sorted(val_paired + val_rna_only)

        if len(train_rna_paired_cells) < MIN_TRAIN_PAIRED_CELLS:
            print(f"[Skip] {ratio_label}: too few paired training cells ({len(train_rna_paired_cells)})")
            continue
        if len(val_rna_query_cells) < MIN_VAL_QUERY_CELLS:
            print(f"[Skip] {ratio_label}: too few validation RNA query cells ({len(val_rna_query_cells)})")
            continue

        # Export task-oriented datasets
        train_rna_all = subset_and_copy(rna, train_rna_all_cells, None)
        train_rna_paired = subset_and_copy(rna, train_rna_paired_cells, None)
        train_protein_paired = subset_and_copy(prot, train_protein_paired_cells, None)

        train_rna_only_adata = subset_and_copy(rna, train_rna_only, None) if len(train_rna_only) > 0 else rna[:0].copy()
        train_protein_only_adata = subset_and_copy(prot, train_prot_only, None) if len(train_prot_only) > 0 else prot[:0].copy()

        val_rna_query = subset_and_copy(rna, val_rna_query_cells, None)
        val_true_protein = subset_and_copy(prot, val_rna_query_cells, None)

        save_split_h5ads(
            outdir=outdir,
            train_rna_all=train_rna_all,
            train_rna_paired=train_rna_paired,
            train_protein_paired=train_protein_paired,
            train_rna_only=train_rna_only_adata,
            train_protein_only=train_protein_only_adata,
            val_rna_query=val_rna_query,
            val_true_protein=val_true_protein,
        )

        split_info = {
            "task": "rna_to_protein_prediction",
            "single_frac": frac,
            "train_cells": train_cells,
            "val_cells": val_cells,
            "train_paired_cells": train_paired,
            "train_rna_only_cells": train_rna_only,
            "train_protein_only_cells": train_prot_only,
            "val_paired_cells": val_paired,
            "val_rna_only_cells": val_rna_only,
            "val_protein_only_cells": val_prot_only,
            "train_rna_all_cells": train_rna_all_cells,
            "train_rna_paired_cells": train_rna_paired_cells,
            "train_protein_paired_cells": train_protein_paired_cells,
            "val_rna_query_cells": val_rna_query_cells,
            "rna_features": list(train_rna_paired.var_names.astype(str)),
            "protein_features": list(train_protein_paired.var_names.astype(str)),
        }
        with open(outdir / "split_info.json", "w", encoding="utf-8") as f:
            json.dump(split_info, f, indent=2, ensure_ascii=False)

        summaries.append({
            "ratio_label": ratio_label,
            "single_frac": frac,
            "train_paired": len(train_paired),
            "train_rna_only": len(train_rna_only),
            "train_protein_only": len(train_prot_only),
            "val_paired": len(val_paired),
            "val_rna_only": len(val_rna_only),
            "val_protein_only": len(val_prot_only),
            "train_rna_all_cells": len(train_rna_all_cells),
            "train_rna_paired_cells": len(train_rna_paired_cells),
            "val_rna_query_cells": len(val_rna_query_cells),
            "rna_features": train_rna_paired.n_vars,
            "protein_features": train_protein_paired.n_vars,
        })

    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(out_root / "summary_all_ratios_rna_to_protein.csv", index=False)
    return summary_df


# =========================
# 4. Main
# =========================
def main():
    set_seed(SEED)
    ensure_dir(OUTPUT_DIR)

    outdir = Path(OUTPUT_DIR)
    rna_qc_path = outdir / "RNA_counts_qc_rnaprot.h5ad"
    prot_qc_path = outdir / "protein_counts_qc.h5ad"
    feature_aligned_path = outdir / "feature_aligned_rna_protein.h5ad"
    protein_map_path = outdir / "protein_gene_name_mapping.csv"

    print("Reading BMMC CITE-seq...")
    rna_raw, prot_raw = read_bmmc_cite(INPUT_H5AD)
    print("RNA raw:", rna_raw.shape)
    print("Protein raw:", prot_raw.shape)

    print("[1/4] Preprocess RNA")
    rna_qc = preprocess_rna(rna_raw)
    safe_write_h5ad(rna_qc, rna_qc_path)

    print("[2/4] Preprocess protein")
    prot_qc = preprocess_protein(prot_raw)

    print("[3/4] Align protein feature names to RNA gene symbols")
    prot_qc, protein_map_df = align_protein_names_to_rna(
        prot=prot_qc,
        rna_var_names=rna_qc.var_names.astype(str),
        hgnc_csv=HGNC_CD_GENE_CSV,
    )
    protein_map_df.to_csv(protein_map_path, index=False)
    safe_write_h5ad(prot_qc, prot_qc_path)

    matched_n = int(protein_map_df["matched_to_rna"].sum()) if "matched_to_rna" in protein_map_df.columns else 0
    print(f"Protein features matched to RNA gene symbols: {matched_n} / {len(protein_map_df)}")

    print("[4/4] Build bookkeeping feature-aligned file")
    feature_aligned = build_feature_aligned(rna_qc, prot_qc)
    safe_write_h5ad(feature_aligned, feature_aligned_path)

    print("Generating RNA -> Protein ratio splits...")
    summary_df = generate_splits(
        rna_qc_path=rna_qc_path,
        prot_qc_path=prot_qc_path,
        out_root=outdir / "results_ratio_loop_rna_to_protein",
        seed=SEED,
        val_frac=VAL_FRAC,
        single_fracs=SINGLE_FRACS,
        rna_keep_prob=RNA_KEEP_PROB,
    )
    print(summary_df)


if __name__ == "__main__":
    main()



Reading BMMC CITE-seq...


/home/zhangye/anaconda3/envs/scmrdr/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


RNA raw: (90261, 13953)
Protein raw: (90261, 134)
[1/4] Preprocess RNA
[2/4] Preprocess protein
[3/4] Align protein feature names to RNA gene symbols
Protein features matched to RNA gene symbols: 102 / 134
[4/4] Build bookkeeping feature-aligned file


/home/zhangye/anaconda3/envs/scmrdr/lib/python3.9/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/zhangye/anaconda3/envs/scmrdr/lib/python3.9/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/zhangye/anaconda3/envs/scmrdr/lib/python3.9/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Generating RNA -> Protein ratio splits...
[Skip] single_100: too few paired training cells (0)
  ratio_label  single_frac  train_paired  train_rna_only  train_protein_only  \
0  single_000          0.0         66632               0                   0   
1  single_020          0.2         53306            6680                6646   
2  single_040          0.4         39979           13281               13372   
3  single_060          0.6         26653           19871               20108   
4  single_080          0.8         13326           26721               26585   

   val_paired  val_rna_only  val_protein_only  train_rna_all_cells  \
0       16658             0                 0                66632   
1       13326          1702              1630                59986   
2        9995          3264              3399                53260   
3        6663          5019              4976                46524   
4        3332          6609              6717                40047   

   